## RL Methods: TD Learning

- The motivation for this is similar to the Monte Carlo approach; this is used when we can't solve the Bellman Equation because transition probabilities and/or rewards are not known

- Unlike Monte Carlo, which relies on the assumption that the task is episodic (i.e. terminates at some point), TD lets you update state value functions and policy incrementally. Therefore, TD is applied to tasks with no natural termination state

- We'll cover 2 forms of TD;
    - **TD(N)**: N-Step TD Learning
    - **TD($\lambda$)**: $\lambda$ Return TD Learning

### N-Step TD

- In the N-Step TD, we want to update either the state value function $V(S_t)$ or action-value function $Q(S_t, a)$ based on some observed reward plus the estimated value of the next state.
    - The $N$ simply tells you how many observed rewards we use in updating our current state value $V(S_t)$

- TD doesn't dictate whether you should learn $Q$ or $V$; it simply gives an approach to learn these by updating an estimate using the difference between successive estimates; or the **TD Error**. This is defined as:

\begin{aligned}
    \delta_t &= R_t + \gamma V(S_{t+1}) - V(S_t)
\end{aligned}

- For example, in `TD(0)`, the updated state-value is the current state-value plus some learning rate multipled by the observed deviation from your current state-value. The observed deviation is the return you experience $R_t + \gamma V(S_{t+1})$ minus the current estimated expected return $V(S_t)$
\begin{aligned}
    V(S_t) &= V(S_t) + \alpha (\hat{G_t^{(1)}} - V(S_t)) \\
    &= V(S_t) + \alpha (R_t + \gamma V(S_{t+1}) - V(S_t))
\end{aligned}
    - Here, we update $V(S_t)$ at every step. So $G_t^{(1)}$ is not known, but your best guess at the time step.
    - This is why some tutorials mention that `TD(N)` approach requires bootstrapping; it comes from the fact that $\hat{G_t^{(1)}} = R_t + \gamma V(S_{t+1})$ is an estimate based on $V(S_{t+1})$
    - This is known as the **1-step bootstrap return**
    - This differs from something like Monte Carlo, for example, where episodes are played out till the end, and we do not need to bootstrap because we use the actual rewards $R_t, R_{t+1}, ...$ to compute our state-value function instead of some intermediate state-value estimate $V(S_{t+1})$

- Extending this idea, for `TD(N)`, we accumulate $N$ steps of the future rewards $R_t, R_{t+1}, ... R_{t+n}$ plus the estimated state value $V(S_{t+n})$ to compute the **TD error** 
\begin{aligned}
    V(S_t) &= V(S_t) + \alpha (\hat{G_t^{(N)}} - V(S_t)) \\
    &= V(S_t) + \alpha (\sum_{i=0}^{n-1} \gamma^{i} R_{t+i} +  \gamma^n V(S_{t+n}) - V(S_t))
\end{aligned}
    - Similar to the example above, $\hat{G_t^{(N)}}$ is known as the **N-step bootstrap return**
    - Fun fact; if N is the episode length, then this is simply Monte Carlo!

- The intuition here is: if $V(S_t)$ has converged, then there should be no TD error observed, the current step return should equal the current state values. So $V(S_t)$ should be unchanged

- Note that the $TD(N)$ idea can be applied to both **continuing** and **episodic** tasks. For clarity, we will implement both here

In [ ]:
import numpy as np
np.random.seed(0)

# Environment setup; we create a number line with 10 states. The right-most state has a reward of 1
# Therefore, train the RL loop to take rightward actions [+1] more than leftward actions [-1]
N_STATES = 10
N_ACTIONS = 5
GAMMA = 0.9
ALPHA = 0.1
EPSILON = 0.5
EPISODES = 1000
TD_N = 5

N_STEPS = 10

TRANSITION_PROBABILITIES = np.random.rand(N_STATES, N_ACTIONS, N_STATES)
TRANSITION_PROBABILITIES /= np.sum(TRANSITION_PROBABILITIES, axis=2, keepdims=True)
REWARDS = np.random.rand(N_STATES, N_ACTIONS, N_STATES) * 5

TERMINAL_STATES = [0, N_STATES-1]

In [ ]:
import numpy as np
from collections import namedtuple

episode_step = namedtuple('episode_step', ['state', 'reward', 'next_state'])

def update_timestep_is_valid(timestep_to_update: int) -> bool:
    return timestep_to_update >= 0

def next_state_is_terminal(next_state: int) -> bool:
    return next_state in [0, N_STATES-1]

def bootstrap_state_is_nonterminal(update_td_return_for_this_timestep, reward_steps_available, termination_time):
    bootstrap_state_index = update_td_return_for_this_timestep + reward_steps_available
    return bootstrap_state_index < termination_time

def n_step_td_episodic(TD_N: int):
    ## Init array to store state-values
    V = np.zeros(N_STATES)

    ## Going across all `EPISODES`
    for _ in range(EPISODES):

        ## Init current state such that it is not a terminal state
        curr_state = np.random.randint(1, N_STATES - 1)
        
        ## For `TD_N`, we need to store N steps of state updates in the trajectory
        trajectory = []
        
        ## For every episode, we define a variable to hold termination time. 
        ## This will be populated in the loop
        termination_time = np.inf
        
        ## Set current episode time as 0
        curr_time = 0  

        ## Playing through the whole episode...
        while True:

            ## If curr_time is less than termination time, we make ONE transition
            if curr_time < termination_time:

                ## Pick an action
                curr_action = np.random.randint(N_ACTIONS)

                ## Make a transition
                next_state = np.random.choice(N_STATES, p=TRANSITION_PROBABILITIES[curr_state, curr_action])

                ## Get the reward of the transition
                reward = REWARDS[curr_state, curr_action, next_state]

                ## Store the transition in the trajectory history
                trajectory.append(episode_step(curr_state, reward, next_state))

                ## If we reach a terminal state
                if next_state_is_terminal(next_state):
                    ## Set termination time condition to the next time step. The condition `curr_time < termination_time` in the episode loop will therefore fail in the next step
                    termination_time = curr_time + 1 
                
                ## Set current state to the next_state
                curr_state = next_state

            # Identify which index we should update. Remember that in TD(N), we need to accumulate N steps in our buffer before updating reward. Therefore, the index we update must always be TD_N - 1 steps behind. For example, to update the first state value index seen in our episode, if TD_N = 3, we need to have made 3 transitions. This happens at curr_time = 2, where we have (0 --> 1, 1 --> 2, 2 --> 3). At this point, we want to update index at 2 - 3 + 1 = 0 
            update_td_return_for_this_timestep = curr_time - TD_N + 1

            ## Assert that the update logic is valid (i.e. your update index >= 0)
            if update_timestep_is_valid(update_td_return_for_this_timestep):
                
                ## Use `G` as the accumulator for the rewards
                G = 0.0
                
                
                ## Depending on the length of the episode, we may not be able to update using the full TD_N update. For example, let's say you want a TD(3) update. Through your simulation, you end up with a trajectory with 5 `episode_steps`, which correspond to the transitions (0 --> 1, 1 --> 2, 2 --> 3, 3 --> 4, 4 --> 5)

                ## For TD(3), we need to have 3 rewards available before updating. This happens in the third timestep. So (0 --> 1, 1 --> 2, 2 --> 3). Therefore at time index 2, we can compute the TD return for the state at time index 0, which also happens to be state 0.

                ## So this update goes on. At 4th timestep, we update time index 1. At 5th timestep, we update time index 2. Now, we reach the time to update time index 3. For this, we need (3 --> 4, 4 --> 5, 5 --> 6). But wait, since 5 state 5 at time index 5 is terminal, we don't have a 5 --> 6!! What do we do?

                ## Instead of discarding usable reward information, TD(N) will make use of as many rewards as possible. Since we cannot do TD(3) without 5 --> 6, we do a TD(2) update instead!

                ## Thus, here we keep track of the number of rewards we have available. When we reach termination, we are at time step 4, and termination_time = 5. But we want to keep updating our state value `V` even if a we don't meet the criteria for TD(3). So for fixed termination time 5, suppose we reach curr_time = 5. Then `update_td_return_for_this_timestep` becomes 5-3+1 = 3. To update index 3, we then say the number of rewards available is min(3, 5-3) = 2
                reward_steps_available = min(
                    TD_N, 
                    termination_time - update_td_return_for_this_timestep
                )

                ## For every reward available
                for i in range(int(reward_steps_available)):
                    ## Add the reward multiplied by the relevant gamma discount
                    G += (GAMMA ** i) * trajectory[update_td_return_for_this_timestep + i].reward

                ## if the time index to update + reward_steps_available is less than termination_time, it means that the next state value is available, because it is non terminal. So we can add the bootstrapped state-value 

                ## BUT if we get update_td_return_for_this_timestep + reward_steps_available == termination_time, then by definition, the last timestep must be terminal. Then there is no need to add the bootstrapped state value of the next state, because there is no "next state" after the terminal state
                if bootstrap_state_is_nonterminal(reward_steps_available=reward_steps_available, update_td_return_for_this_timestep=update_td_return_for_this_timestep, termination_time=termination_time):

                    ## After adding N rewards, add the bootstrapped state value of the next state. This only applies if we have not reached terminal state. Otherwise the value of the next state is 0 anyway, because there is no post-terminal transition
                    
                    G += (GAMMA ** reward_steps_available) * V[
                        trajectory[
                            update_td_return_for_this_timestep + reward_steps_available - 1
                        ].next_state
                    ]

                # Update state value
                state_to_update = trajectory[update_td_return_for_this_timestep].state
                V[state_to_update] += ALPHA * (G - V[state_to_update])

            curr_time += 1

            # Break out of the while loop if our update value is termination_time - 1 (because there are no further rewards after this step)
            if update_td_return_for_this_timestep == termination_time - 1:
                break

    return V


In [ ]:
n_step_td_episodic(1)

In [ ]:
from collections import deque, namedtuple
import numpy as np

episode_step = namedtuple('episode_step', ['state', 'reward', 'next_state'])

def n_step_td_continuing(steps: int = N_STEPS):
    """
    N-step TD for continuing tasks.
    """

    V = np.zeros(N_STATES, dtype=float)

    # Store 
    td_n_buffer = deque(maxlen=TD_N)

    # Start at a random (non-terminal) state
    curr_state = np.random.randint(1, N_STATES - 1)

    for _ in range(steps):
        # Random action (ε-greedy or random for continuing case)
        action = np.random.randint(N_ACTIONS)

        # Sample next state using transition probabilities
        next_state = np.random.choice(N_STATES, p=TRANSITION_PROBABILITIES[curr_state, action])

        # Sample reward for (s, a, s_next)
        reward = REWARDS[curr_state, action, next_state]

        # Append to buffers
        td_n_buffer.append(episode_step(state=curr_state, reward=reward, next_state=next_state))

        # When we have enough samples, perform an update
        G = 0
        if len(td_n_buffer) == TD_N:
            for t, step in enumerate(td_n_buffer):
                G += (GAMMA ** t) * step.reward
            G += (GAMMA ** TD_N) * V[td_n_buffer[-1].next_state]  # bootstrap from S_{t+n}

            state_to_update = td_n_buffer[0].state
            V[state_to_update] += ALPHA * (G - V[state_to_update])

            # Slide window manually
            td_n_buffer.popleft()

        curr_state = next_state

    return V

In [ ]:
n_step_td_continuing(20000)

### $\lambda$ Return TD

- Having discussed $TD(N)$, we now move on to $TD(\lambda)$. This is very similar to the $TD(N)$ case, we want to use successive TD errors to update either the state value $V(s)$ or action value $Q(s,a)$

- Recall in $TD(N)$ that we define some $N$ steps of reward accumulation before bootstrapping an update

\begin{aligned}
    G^{(n)}_t &= R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})
\end{aligned}

- The question arises; what value of $N$ do we choose? 
    - If $N$ is small, we update fast, and therefore the variance is low, because we each step is small. But because we don't accumulate the update over multiple observations of rewards, the estimate is more biased
    - If $N$ is large, we update slowly averaging over more reward steps. Thus, the bias is lower. But since each update takes longer to accumulate, the variance is also larger

- To resolve this problem of choosing $N$, we introduce $TD(\lambda)$

- $TD(\lambda)$ uses **all** possible future steps without bootstrapping, but **exponentially weights** them using $\lambda \in [0, 1)$. The idea here is that you don't need to choose which steps are important; since you are using all of them, they are all important. You just determine how best to weight them using $\lambda$

- Formally, the $TD(\lambda)$ setup is:

\begin{aligned}
    G^{(\lambda)}_t &= (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}
\end{aligned}

- Why geometric weighting? Because geometric weights are guaranteed to sum to 1. Looking at the weights:

\begin{aligned}
    w_n &= (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \\
    &= (1 - \lambda) \sum_{n=0}^{\infty} \lambda^{n} \\
    &= (1 - \lambda) \cdot \frac{1}{1 - \lambda} & \text{by geometric series} \\
    &= 1
\end{aligned}

- Since the geometric weights only sum to 1 in the limit, $TD(\lambda)$ should conceptually only be valid for continuous learning tasks, not episodic task. In theory, you probably could apply it to episodic tasks, and lose the "weighted average" interpretation of the weights. But for the purposes of accuracy, we'll focus on the continuous learning task

#### General form of $TD(\lambda)$

- Let $G_t(N)$ be the $TD(N)$ return
\begin{aligned}
    G_t(1) &= R_{t+1} + \gamma V(S_{t+1}) \\
    G_t(2) &= R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+3}) \\\
    G_t(3) &= R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V(S_{t+4})
    ...
\end{aligned}

- Recall that in $TD(\lambda)$, we are simply taking the exponentially weighted sums of the $TD(N)$ returns. Therefore:

\begin{aligned}
    G_t(\lambda) &= (1 - \lambda) \cdot [\lambda^0 G_t(1) + \lambda^1 G_t(2) + \lambda^2 G_t(3) + ...] \\
    &= (1 - \lambda) \cdot [ R_{t+1} + \gamma V(S_{t+1}) + \lambda (R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2})) + \lambda^2 (R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V(S_{t+3})) + ...] \\
    &= (1 - \lambda) \cdot [ (R_{t+1} + \lambda R_{t+1} + \lambda^2 R_{t+1} + ...) + (\lambda \gamma R_{t+2} + \lambda^2 \gamma R_{t+2} + ...) + (\lambda^2 \gamma^2 R_{t+3} + \lambda^3 \gamma^2 R_{t+3} + ...) + ... + (\gamma V_{s+1} + \lambda \gamma^2 V_{s+2} + \lambda^2 \gamma^3 V_{s+3} + ...)] \\
\end{aligned}

- Let's first focus on the weights $W(R_{t+k})$ of the reward terms:

\begin{aligned}
    W(R_{t+1}) &= (1 - \lambda) [1 + \lambda + \lambda^2 + ...] \\
    &= (1 - \lambda) \frac{1}{1 - \lambda} \\
    &= 1 \\ \\

    W(R_{t+2}) &= (1 - \lambda) [\lambda \gamma + \lambda^2 \gamma + \lambda^3 \gamma ...] \\
    &= (1 - \lambda) \frac{\lambda \gamma}{1 - \lambda} \\
    &= \lambda \gamma \\ \\

    W(R_{t+3}) &= (1 - \lambda) [\lambda^2 \gamma^2 + \lambda^3 \gamma^2 + \lambda^4 \gamma^2 ...] \\
    &= (1 - \lambda) \frac{\lambda^2 \gamma^2}{1 - \lambda} \\
    &= \lambda^2 \gamma^2 \\ \\
\end{aligned}

- From the pattern above, for a given $R_{t+k}$, we see that as $\lambda \rightarrow 1$, we will get $(\lambda \gamma)^{k-1} = \gamma^{k-1}$

\begin{aligned}
    W(R_{t+k}) &= (1 - \lambda) [\lambda^{k-1} \gamma^{k-1} + \lambda^{k} \gamma^{k-1} + \lambda^{k+1} \gamma^{k-1} + ...] \\
    &= (1 - \lambda) \gamma^{k-1} \sum_{i=k-1}^{\infty} \lambda^i \\
    &= (1 - \lambda) \gamma^{k-1} \frac{\lambda^{k-1}}{1 - \lambda} \\
    &= (\gamma \lambda)^{k-1} 
\end{aligned}

- Next, we focus on the weight on the bootstrap term for each return $G^{(n)}_t$, which takes the general form $W_V(S_{t+k})$. Recall that we previously accounted for all reward weights, so we collect the $1 - \lambda$ term into the weight of the bootstrap term:

\begin{aligned}
    W_V(S_{t+k}) &= (1 - \lambda) \lambda^{k-1} \gamma^{k} \\
\end{aligned}

- As such, we arrive at the key $TD(\lambda)$ formulation:

\begin{aligned}
    G_t(\lambda) &= (1 - \lambda) \cdot [\lambda^0 G_t(1) + \lambda^1 G_t(2) + \lambda^2 G_t(3) + ...] \\
    &= (1 - \lambda) \cdot [ R_{t+1} + \gamma V(S_{t+1}) + \lambda (R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2})) + \lambda^2 (R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V(S_{t+3})) + ...] \\
    &= (1 - \lambda) \cdot [ (R_{t+1} + \lambda R_{t+1} + \lambda^2 R_{t+1} + ...) + (\lambda \gamma R_{t+2} + \lambda^2 \gamma R_{t+2} + ...) + (\lambda^2 \gamma^2 R_{t+3} + \lambda^3 \gamma^2 R_{t+3} + ...) + ... + (\gamma V_{s+1} + \lambda \gamma^2 V_{s+2} + \lambda^2 \gamma^3 V_{s+3} + ...)] \\
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1}  R_{t+k} + (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \gamma^n V(S_{t+n})
\end{aligned}

#### Boundary values of $\lambda$

- It's useful to see what happens to the formulation at the boundary values of $\lambda$

- If $\lambda = 0$, all powers of $\lambda^{n-1} = 0$, except for the case $0^0 = 1$. So $TD(\lambda)$ reduces to a $TD(N=0)$ return
\begin{aligned}
    G^{(\lambda)}_t &= (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)} \\
    &= \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)} \\
    &= G_t^{(1)} \\
    &= R_{t+1} + \gamma V(S_{t+1})
\end{aligned}

- If $\lambda \rightarrow 1$, in the limit, $TD(\lambda=1)$ simply becomes Monte Carlo, where we take the full episodic reward

\begin{aligned}
G_t(\lambda) &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1}  R_{t+k} + (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \gamma^n V(S_{t+n}) \\
&= \sum_{k=1}^{\infty} (\gamma)^{k-1}  R_{t+k} \\
&= R_{t+1} + \gamma R_{t+2} + ...
\end{aligned}

#### Backward vs Forward Views

- Having gone through the theory of $TD(\lambda)$, there is an immediate gotcha when implementing - it's not possible to actually accumulate infinite reward terms to compute $G$

\begin{aligned}
    G_t(\lambda) &= (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t(n)
\end{aligned}

- Let's consider another view of this. Recall that in TD learning, we are trying to incrementally update some value function by the TD error:

\begin{aligned}
    V(S_t) \leftarrow V(S_t) + \alpha [G_t(\lambda) - V(S_t)]
\end{aligned}

- Let's call $G_t(\lambda) - V(S_t)$ the $\lambda$ return error

- **CLAIM:** Since we cannot compute $G_t(\lambda)$ by taking the infinite sum of $TD(N)$ terms (the **forward view**), we can replace the $\lambda$ return error with the weighted sum to infinity of one step TD errors (**backward view**):

\begin{aligned}
    G_t(\lambda) - V(S_t) &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k} \\ \\

    \delta_{t+k} &= R_{t+k} + \gamma V(S_{t+k+1}) - V(S_{t+k})
\end{aligned}

- We previously said that we cannot accumulate infinite rewards before computing $G_t(\lambda)$. But isn't this just another sum to infinity?
    - Yes, it is. So this also cuts off at some arbitrary point. 
    - But unlike summing the TD(N) terms, we don't need a buffer to accumulate rewards in this approach! 
    - Since updates are happening for every $\delta_{t+k}$, and $\delta_{t+k}$ can be computed at every step, there is no delay in the updating of the state value at time $t$, so long as we can accept bootstrapped values of the next state $V(S_{t+k+1})$

##### Proof that backward = forward view


- Let's start with definitions:

\begin{aligned}
    \delta_t &= R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \\
    G_t(n) &= \sum_{j=1}^{n} \gamma^{j-1} R_{t+j} + \gamma^n V(S_{t+n}) \\
    G_t(\lambda) &= (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t(n) \\ 
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1}  R_{t+k} + (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \gamma^n V(S_{t+n}) & \text{Forward View}
\end{aligned}

- Looking at the RHS of the equation to prove, let's call the summation of discounted 1-step TD errors $S$. So we have

\begin{aligned}
    S &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k}
\end{aligned}

- Now, by the definition above $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$

\begin{aligned}
    S &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k} \\
    &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} [R_{t+k+1} + \gamma V(S_{t+k+1}) - V(S_{t+k})] \\
    &= \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k R_{t+k+1}}_{(A)} + 
    \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k \gamma V(S_{t+k+1})}_{(B)} - 
    \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k V(S_{t+k})}_{(C)}
\end{aligned}

- Let's rewrite $(C)$ by separating out the first term of the summation, when $k=0$

\begin{aligned}
    (C) &= \sum_{k=0}^{\infty} (\gamma \lambda)^k V(S_{t+k}) \\
    &= V(S_{t+k}) + \sum_{k=1}^{\infty} (\gamma \lambda)^k V(S_{t+k}) \\
\end{aligned}

- Then, rewriting $S$

\begin{aligned}
    S &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k} \\
    &= (A) + (B) - (C) \\
    &= (A) + \underbrace{(B) - \sum_{k=1}^{\infty} (\gamma \lambda)^k V(S_{t+k})}_{(D)} - V(S_{t+k})
\end{aligned}

- Notice that the term we extracted can be combined with $(B)$, which leads to 

\begin{aligned}
    (D) &= \sum_{k=0}^{\infty} (\gamma \lambda)^k \gamma V(S_{t+k+1}) - \sum_{k=1}^{\infty} (\gamma \lambda)^k V(S_{t+k}) \\
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1} \gamma V(S_{t+k}) - \sum_{k=1}^{\infty} (\gamma \lambda)^k V(S_{t+k}) \\
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1} \gamma V(S_{t+k}) - (\gamma \lambda)^k V(S_{t+k}) \\
    &= \sum_{k=1}^{\infty} V(S_{t+k}) [(\gamma \lambda)^{k-1} \gamma - (\gamma \lambda)^k] \\
    &= \sum_{k=1}^{\infty} V(S_{t+k}) (\gamma \lambda)^{k-1} [\gamma - \gamma \lambda] \\
    &= \sum_{k=1}^{\infty} V(S_{t+k}) (\gamma \lambda)^{k-1} \gamma (1 - \lambda) \\
    &= (1 - \lambda) \sum_{k=1}^{\infty} \gamma^k \lambda^{k-1} V(S_{t+k}) \\
\end{aligned}

- Looking at the forward view, this is exactly the bootstrap term!

- Simiarly, let's rewrite $(A)$

\begin{aligned}
    (A) &= \sum_{k=0}^{\infty} (\gamma \lambda)^k R_{t+k+1} \\
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1} R_{t+k} \\
\end{aligned}

- Going back to the forward view again, this is exactly the discounted reward term!!

- So going back to $S$

\begin{aligned}
    S &= \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k} \\
    &= \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k R_{t+k+1}}_{(A)} + 
    \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k \gamma V(S_{t+k+1})}_{(B)} - 
    \underbrace{\sum_{k=0}^{\infty} (\gamma \lambda)^k V(S_{t+k})}_{(C)} \\
    &= \sum_{k=1}^{\infty} (\gamma \lambda)^{k-1} R_{t+k} + (1 - \lambda) \sum_{k=1}^{\infty} \gamma^k \lambda^{k-1} V(S_{t+k}) - V(S_{t+k}) \\ \\

    &\therefore V(S_{t+k}) + S = G_{t+k}(\lambda) \\
    G_{t+k}(\lambda) &= V(S_{t+k}) + \sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k}
\end{aligned}

- This tells us that, though the forward view asks us to accumulate $\infty$ rewards before updating the current state $t$, it is actually equivalent to taking the current state value, and adding discounted 1-step TD errors for infinite steps!

- The main difference here is that, of course, we no longer need to "buffer". The state values can be updated incrementally!

- This is why, to update state value, we can simply add the summation of 1 step TD errors to the existing state value

\begin{aligned}
    V(S_t) \leftarrow V(S_t) + \alpha [G_t(\lambda) - V(S_t)] \\
    V(S_t) \leftarrow V(S_t) + \alpha [\sum_{k=0}^{\infty} (\gamma \lambda)^{k} \delta_{t+k}] \\
\end{aligned}

- And we don't need to wait for $\infty$ steps before updating, because at every step, I simply need to attribute the 1-step TD error proportionally to every state $S_t$! 
    - That is, suppose I have states 1,2,3 and I move from 1 --> 2 --> 3
    - Then 
        - the TD error I get from step 2 will be added to V(S_2) with weight $(\gamma \lambda)^0$
        - the TD error I get from step 2 will be added to V(S_1) with weight $(\gamma \lambda)^1$

- The idea here is that at every step, I update all prior visited states. Just scaling the error by the number of timesteps ago I visited the state

- To maintain the weights, we simply need to maintain the product of $(\gamma \lambda)^x$ for every state, and add 1 for the latest state visited. Then we just multiply the entire array by a factor of $(\gamma \lambda)$ in each step
    - This is known as the **eligibility trace**


#### Implementation

In [ ]:
import numpy as np
np.random.seed(0)

N_STATES = 10
N_ACTIONS = 5
GAMMA = 0.9
ALPHA = 0.1
EPISODES = 1000
TD_LAMBDA = 0.5

N_STEPS = int(1e5)

TRANSITION_PROBABILITIES = np.random.rand(N_STATES, N_ACTIONS, N_STATES)
TRANSITION_PROBABILITIES /= np.sum(TRANSITION_PROBABILITIES, axis=2, keepdims=True)
REWARDS = np.random.rand(N_STATES, N_ACTIONS, N_STATES) * 5

In [ ]:
from collections import deque, namedtuple

episode_step = namedtuple('episode_step', ['curr_state', 'action', 'reward', 'next_state'])

def td_lambda_continuing():
    V = np.zeros(N_STATES)
    curr_state = np.random.randint(N_STATES)
    ELIGIBILITY_TRACE = np.zeros(N_STATES)
    for _ in range(N_STEPS):
        action = np.random.randint(N_ACTIONS)
        next_state = np.random.choice(N_STATES, p=TRANSITION_PROBABILITIES[curr_state, action])
        curr_reward = REWARDS[curr_state, action, next_state]

        one_step_td_error = curr_reward + (GAMMA * V[next_state]) - V[curr_state]

        ELIGIBILITY_TRACE[curr_state] += 1
        V[curr_state] += ALPHA * one_step_td_error
        
        ELIGIBILITY_TRACE *= TD_LAMBDA * GAMMA
        curr_state = next_state
    
    return V

td_lambda_continuing()